# Step 8 — Fine-tuning the Predictive Analyst (Unsloth QLoRA) ⭐ *critical feature*

> **Runs on the AMD MI300X node only** (`requirements-gpu.txt`), **not** on the CPU dev machine.
> This notebook is the recipe + eval harness; do not execute it here. See `docs/finetuning.md`.

**Goal:** move question-prediction skill into the weights -> higher recall@10 **and** ~60-70%
fewer prompt tokens (drop few-shot examples). Serve the LoRA adapter via vLLM `--enable-lora`.

## 1. Build the training set
Pairs of *(prepared remarks + financial snapshot)* -> *(actual analyst Q&A turns)* mined from
earnings transcripts: HF `lamini/earnings-calls-qa`, `jlh-ibm/earnings_call`, Kaggle Motley-Fool.

In [ ]:
# GPU node only.
from datasets import load_dataset
ds = load_dataset("lamini/earnings-calls-qa", split="train")

def to_pair(row):
    context = row.get("transcript") or row.get("context") or ""
    question = row.get("question") or ""
    return {"text": f"<context>\n{context}\n</context>\nLikely analyst question:\n{question}"}

train = ds.map(to_pair, remove_columns=ds.column_names)
print(train[0]["text"][:300])

## 2. QLoRA fine-tune with Unsloth
Primary path is Unsloth; if its ROCm kernels are unavailable, the identical LoRA config runs via
`trl` + `peft` (`docs/finetuning.md`). Adapter format + vLLM serving are unchanged either way.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

model, tok = FastLanguageModel.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B", max_seq_length=8192, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=train, dataset_text_field="text",
    args=TrainingArguments(per_device_train_batch_size=2, gradient_accumulation_steps=4,
                           warmup_steps=10, num_train_epochs=1, learning_rate=2e-4,
                           fp16=False, bf16=True, logging_steps=10, output_dir="models/qpredict"))
trainer.train()
model.save_pretrained("models/lora_qpredict")   # serve via: vllm serve ... --enable-lora

## 3. Evaluate (base vs fine-tuned) — the headline slide
Held-out companies/quarters. Report **recall@10** and **semantic match** of predicted vs real
analyst questions, plus **mean input tokens/call** (few-shot 6 -> 0). Ship the winner via
`--enable-lora`; the base model stays the safe default until then.

In [ ]:
# Pseudocode for the eval harness (GPU node):
#   base_qs  = predict_with(base_model,  context, few_shot=6)
#   tuned_qs = predict_with(tuned_model, context, few_shot=0)
#   recall@10 = overlap(tuned_qs, held_out_real_qs)
#   tokens_saved = mean_input_tokens(base) - mean_input_tokens(tuned)
print("See docs/evaluation.md for the metric definitions and targets.")

### Stretch — finance-tuned embedding model
A drop-in `sentence-transformers` contrastive fine-tune of `bge-base` on financial-jargon pairs
lifts retrieval hit-rate across the whole RAG pipeline. See
`docs/finetuning.md#secondary-stretch-fine-tune-the-embedding-model`.